In [1]:
!uv --version

uv 0.13.5 (46776a95d 2026-10-21 x86_64-pc-windows-msvc)


In [3]:
!type pyproject.toml

[project]
name = "mlzoomcamp-2026-deployment"
version = "0.1.0"
description = "Reference deployment project for ML Zoomcamp 2026"
requires-python = ">=3.11,<3.12"
dependencies = [
    "fastapi==0.119.1",
    "numpy==2.3.3",
    "scikit-learn==1.7.2",
    "uvicorn==0.38.0",
]

[tool.uv]
exclude-newer = "2026-09-14T23:59:59Z"


In [4]:
import pickle 

dv, model = pickle.load(open('pipeline.bin', 'rb'))

lead = {
  "lead_source": "paid_ads",
  "industry": "technology",
  "employment_status": "employed",
  "location": "north_america",
  "number_of_courses_viewed": 2,
  "annual_income": 79276.0,
  "interaction_count": 4,
  "lead_score": 0.41
}

X = dv.transform(lead)
y = model.predict_proba(X)
print(round(y[0, 1],3))

0.533


In [2]:
import json
import requests

payload = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

with open("payload.json", "w") as f:
    json.dump(payload, f)

r = requests.post("http://localhost:9696/predict", json=payload)
print(r.status_code)
print(r.json())

200
{'conversion_probability': 0.769799, 'conversion': True}


In [3]:
print(open("Dockerfile").read())

FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3

COPY --from=ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4 /uv /uvx /bin/

ENV PATH="/code/.venv/bin:$PATH" \
    UV_COMPILE_BYTECODE=1 \
    UV_LINK_MODE=copy

WORKDIR /code
COPY pyproject.toml uv.lock .python-version ./
RUN uv sync --locked --no-dev

COPY model.py predict.py feature_defaults.json model_metadata.json pipeline.bin q6_test.py ./

EXPOSE 9696
ENTRYPOINT ["uvicorn", "predict:app", "--host", "0.0.0.0", "--port", "9696"]



In [4]:
!python q6_test.py

{'conversion_probability': 0.769799, 'conversion': True}


In [5]:
r = requests.get("http://localhost:9696/health")
print(r.status_code)
print(r.json())

200
{'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '5cb9195a7a160fdc91a1b8f57172055e33ced1994870742b937d5273f4021c67'}
